In [1]:
# ══════════════════════════════════════════════════════════════════════════════
# PEMMSS INPUT FILE GENERATION
# Takes Global_Graphite_Projects_final.csv and creates:
#   1. input_projects.csv: existing projects for PEMMSS
#   2. input_exploration_production_factors.csv: distributions for new discoveries
# ══════════════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np

df = pd.read_csv('./outputs/Global_Graphite_Projects_final.csv')
print(f"Loaded {len(df)} records across {df['Country (Short Form)'].nunique()} countries")

Loaded 253 records across 33 countries


In [2]:
# ══════════════════════════════════════════════════════════════════════════════
# 0. CLEAN-UP AND DEDUPLICATION
# ══════════════════════════════════════════════════════════════════════════════

# Fix "North Korea" / "Korea, North" duplication
df.loc[df['Country (Short Form)'] == 'North Korea', 'Country (Short Form)'] = 'Korea, North'

# Drop records with Status = Inactive (not useful for PEMMSS projection)
n_before = len(df)
df = df[df['Status'] != 'Inactive'].reset_index(drop=True)
print(f"Dropped {n_before - len(df)} Inactive records, {len(df)} remaining")
print(f"Countries: {df['Country (Short Form)'].nunique()}")

Dropped 16 Inactive records, 237 remaining
Countries: 32


# 1. Build input projects

In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# 1. BUILD input_projects.csv
# ══════════════════════════════════════════════════════════════════════════════

# ── 1.1 Status mapping ──
# PEMMSS: 2=Already Producing, 1=Developed/Not Yet Producing, 0=Undeveloped
status_map = {
    'Production': 2,
    'Development': 1,
    'Feasibility': 1,
    'Exploration': 0,
    'Care and Maintenance': 1,  # infrastructure exists, temporarily idle
}

# ── 1.2 Development probability by status ──
# Probability that the project proceeds to production
dev_prob_map = {
    'Production': 1.0,
    'Development': 0.50,
    'Feasibility': 0.25,
    'Exploration': 0.10,
    'Care and Maintenance': 0.60,  # restart likely
}

# ── 1.3 Build the projects DataFrame ──
projects = pd.DataFrame()

projects['P_ID_NUMBER'] = range(len(df))
projects['NAME'] = df['Name'].values
projects['REGION'] = df['Country (Short Form)'].values
projects['LATITUDE'] = df['DD Latitude'].values
projects['LONGITUDE'] = df['DD Longitude'].values
projects['DEPOSIT_TYPE'] = 'Flake graphite'
projects['COMMODITY'] = 'graphite'

# Resource: total ore tonnage
projects['REMAINING_RESOURCE'] = df['Total Resource (mt)'].values

# Capacity: concentrate t/yr (pass directly)
cap = df['Annual Production Capacity'].values.copy()
cap = np.where(cap < 0, np.nan, cap)  # remove -999 sentinels
projects['PRODUCTION_CAPACITY'] = cap

# Status
projects['STATUS'] = df['Status'].map(status_map).values

# Timing
projects['DISCOVERY_YEAR'] = np.nan
projects['START_YEAR'] = df['Project Active'].values

# Development probability
projects['DEVELOPMENT_PROBABILITY'] = df['Status'].map(dev_prob_map).values

# Grade: TGC% as mass fraction
grade_pct = df['Resource Grade (TGC%)'].values
projects['GRADE'] = grade_pct / 100  # 5% -> 0.05

# Recovery
projects['RECOVERY'] = 0.85

# Brownfield factors
projects['BROWNFIELD_TONNAGE_FACTOR'] = 0.05
projects['BROWNFIELD_GRADE_FACTOR'] = 1

# Value (not pre-computed)
projects['VALUE_NET'] = np.nan
projects['VALUE_RECOVERY_NET'] = np.nan

# Mine cost model — placeholder
projects['MINE_COST_MODEL'] = 'fixed'
projects['MINE_COST_A'] = 0
projects['MINE_COST_B'] = 0
projects['MINE_COST_C'] = 0
projects['MINE_COST_D'] = 0

# Revenue model — placeholder
projects['REVENUE_MODEL'] = 'contained_value'
projects['REVENUE_A'] = 0
projects['REVENUE_B'] = 0
projects['REVENUE_C'] = 0
projects['REVENUE_D'] = 0

# Cost model — placeholder
projects['COST_MODEL'] = 'contained_value'
projects['COST_A'] = 0
projects['COST_B'] = 0
projects['COST_C'] = 0
projects['COST_D'] = 0

print(f"Built input_projects: {len(projects)} rows, {projects.columns.size} columns")
print(f"\nStatus distribution:")
for s, code in sorted(status_map.items(), key=lambda x: -x[1]):
    n = (projects['STATUS'] == code).sum()
    print(f"  {s:25s} (code {code}): {n:4d} projects")

print(f"\nData completeness:")
print(f"  With REMAINING_RESOURCE: {projects['REMAINING_RESOURCE'].notna().sum()}/{len(projects)}")
print(f"  With PRODUCTION_CAPACITY: {projects['PRODUCTION_CAPACITY'].notna().sum()}/{len(projects)}")
print(f"  With GRADE: {projects['GRADE'].notna().sum()}/{len(projects)}")
print(f"  With coordinates: {projects['LATITUDE'].notna().sum()}/{len(projects)}")

Built input_projects: 237 rows, 34 columns

Status distribution:
  Production                (code 2):   31 projects
  Development               (code 1):   32 projects
  Feasibility               (code 1):   32 projects
  Care and Maintenance      (code 1):   32 projects
  Exploration               (code 0):  174 projects

Data completeness:
  With REMAINING_RESOURCE: 96/237
  With PRODUCTION_CAPACITY: 41/237
  With GRADE: 97/237
  With coordinates: 197/237


In [4]:
# Save input_projects.csv

projects.to_csv('./outputs/input_projects.csv', index=False)
print(f"Saved input_projects.csv: {len(projects)} rows")
print(projects.head(10).to_string())

Saved input_projects.csv: 237 rows
   P_ID_NUMBER                       NAME      REGION   LATITUDE  LONGITUDE    DEPOSIT_TYPE COMMODITY  REMAINING_RESOURCE  PRODUCTION_CAPACITY  STATUS  DISCOVERY_YEAR  START_YEAR  DEVELOPMENT_PROBABILITY   GRADE  RECOVERY  BROWNFIELD_TONNAGE_FACTOR  BROWNFIELD_GRADE_FACTOR  VALUE_NET  VALUE_RECOVERY_NET MINE_COST_MODEL  MINE_COST_A  MINE_COST_B  MINE_COST_C  MINE_COST_D    REVENUE_MODEL  REVENUE_A  REVENUE_B  REVENUE_C  REVENUE_D       COST_MODEL  COST_A  COST_B  COST_C  COST_D
0            0               Gallois Mine  Madagascar -19.230198  48.945072  Flake graphite  graphite                 NaN             150000.0       2             NaN      2017.0                     1.00     NaN      0.85                       0.05                        1        NaN                 NaN           fixed            0            0            0            0  contained_value          0          0          0          0  contained_value       0       0       0       0

# 2. Exploration production factors

In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# 2. FIT TONNAGE AND GRADE DISTRIBUTIONS
#
# PEMMSS uses random.lognormvariate(mu, sigma) where:
#   mu = mean of ln(x)
#   sigma = std of ln(x)
#   max = cap value (GRADE_C / TONNAGE_C)
#
# Strategy:
#   - For countries with >=3 deposits with resource data: fit from their data
#   - For countries with <3 deposits: use pooled global distribution
#   - Grade is fitted as mass fraction (e.g., 0.05 for 5% TGC)
#   - Tonnage is fitted as ore tonnes
# ══════════════════════════════════════════════════════════════════════════════

# ── 2.1 Prepare data for fitting ──
res = df[df['Total Resource (mt)'].notna() & (df['Total Resource (mt)'] > 0)].copy()
res['grade_frac'] = res['Resource Grade (TGC%)'] / 100

print(f"Deposits with resource data for fitting: {len(res)}")
print(f"  With grade: {res['grade_frac'].notna().sum()}")
print(f"  Countries: {res['Country (Short Form)'].nunique()}")


def fit_lognormal(values, label=""):
    """Fit lognormal parameters from observed data."""
    clean = values.dropna()
    clean = clean[clean > 0]
    if len(clean) < 2:
        return None
    ln_vals = np.log(clean)
    mu = ln_vals.mean()
    sigma = ln_vals.std(ddof=1)  # sample std
    max_val = clean.max()
    if sigma == 0:
        sigma = 0.1  # avoid zero dispersion
    return {'mu': mu, 'sigma': sigma, 'max': max_val, 'n': len(clean),
            'median': np.exp(mu), 'mean': np.exp(mu + sigma**2/2)}


# ── 2.2 Global (pooled) distributions ──
global_tonnage = fit_lognormal(res['Total Resource (mt)'], 'global_tonnage')
global_grade = fit_lognormal(res['grade_frac'], 'global_grade')

print(f"\n--- Global pooled distributions ---")
print(f"Tonnage (ore mt): mu={global_tonnage['mu']:.3f}, sigma={global_tonnage['sigma']:.3f}, "
      f"max={global_tonnage['max']/1e6:.0f} Mt, n={global_tonnage['n']}, "
      f"median={global_tonnage['median']/1e6:.1f} Mt")
print(f"Grade (fraction): mu={global_grade['mu']:.3f}, sigma={global_grade['sigma']:.3f}, "
      f"max={global_grade['max']:.3f}, n={global_grade['n']}, "
      f"median={global_grade['median']*100:.1f}%")

Deposits with resource data for fitting: 96
  With grade: 96
  Countries: 19

--- Global pooled distributions ---
Tonnage (ore mt): mu=16.531, sigma=1.916, max=1576 Mt, n=96, median=15.1 Mt
Grade (fraction): mu=-2.549, sigma=0.583, max=0.368, n=96, median=7.8%


In [6]:
# ── 2.3 Per-country distributions ──

MIN_FIT_N = 3  # minimum deposits to fit country-specific distribution

countries = sorted(df['Country (Short Form)'].unique())
country_params = {}

print(f"{'Country':<20} {'n_res':>5} {'Tonnage mu':>10} {'Ton sigma':>10} "
      f"{'Grade mu':>10} {'Grd sigma':>10} {'Source':<10}")
print("-" * 80)

for country in countries:
    c_res = res[res['Country (Short Form)'] == country]
    
    t_fit = fit_lognormal(c_res['Total Resource (mt)'])
    g_fit = fit_lognormal(c_res['grade_frac'])
    
    # Use country-specific if enough data, else global
    if t_fit and t_fit['n'] >= MIN_FIT_N:
        tonnage_params = t_fit
        t_source = 'country'
    else:
        tonnage_params = global_tonnage.copy()
        t_source = 'global'
    
    if g_fit and g_fit['n'] >= MIN_FIT_N:
        grade_params = g_fit
        g_source = 'country'
    else:
        grade_params = global_grade.copy()
        g_source = 'global'
    
    source = 'country' if t_source == 'country' or g_source == 'country' else 'global'
    
    country_params[country] = {
        'tonnage': tonnage_params,
        'grade': grade_params,
        't_source': t_source,
        'g_source': g_source,
    }
    
    print(f"{country:<20} {len(c_res):>5} {tonnage_params['mu']:>10.3f} "
          f"{tonnage_params['sigma']:>10.3f} {grade_params['mu']:>10.3f} "
          f"{grade_params['sigma']:>10.3f} {source:<10}")

Country              n_res Tonnage mu  Ton sigma   Grade mu  Grd sigma Source    
--------------------------------------------------------------------------------
Afghanistan              0     16.531      1.916     -2.549      0.583 global    
Australia               18     16.684      1.401     -2.483      0.415 country   
Bhutan                   0     16.531      1.916     -2.549      0.583 global    
Brazil                   1     16.531      1.916     -2.549      0.583 global    
Canada                  12     16.785      1.550     -2.876      0.801 country   
Chad                     0     16.531      1.916     -2.549      0.583 global    
China                    3     18.144      1.903     -2.471      0.732 country   
Ethiopia                 0     16.531      1.916     -2.549      0.583 global    
Greenland                1     16.531      1.916     -2.549      0.583 global    
Guinea                   1     16.531      1.916     -2.549      0.583 global    
India            

In [7]:
# ── 2.4 Fit capacity distribution (global) ──
# Used for CAPACITY_A, CAPACITY_B, CAPACITY_SIGMA in exploration_production_factors

cap_data = df[df['Annual Production Capacity'] > 0]['Annual Production Capacity']
cap_fit = fit_lognormal(cap_data)

print(f"--- Capacity distribution (concentrate t/yr) ---")
print(f"  n={cap_fit['n']}, mu={cap_fit['mu']:.3f}, sigma={cap_fit['sigma']:.3f}")
print(f"  Median: {cap_fit['median']:,.0f} t/yr")
print(f"  Mean: {cap_fit['mean']:,.0f} t/yr")
print(f"  Max observed: {cap_fit['max']:,.0f} t/yr")

--- Capacity distribution (concentrate t/yr) ---
  n=41, mu=10.073, sigma=1.628
  Median: 23,694 t/yr
  Mean: 89,179 t/yr
  Max observed: 356,000 t/yr


In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# 3. BUILD input_exploration_production_factors.csv
#
# One row per country (REGION = country).
# Single DEPOSIT_TYPE = "Flake graphite" for all.
# ══════════════════════════════════════════════════════════════════════════════

# ── 3.1 Calculate development probability per country ──
# Based on ratio of advanced projects (Production + Development) to total

def calc_dev_probability(country_df):
    """Development probability for new discoveries in this country."""
    total = len(country_df)
    advanced = len(country_df[country_df['Status'].isin(['Production', 'Development'])])
    if total == 0:
        return 0.10  # default
    # Base rate with a floor
    prob = max(advanced / total, 0.05)
    return round(prob, 4)


# ── 3.2 Calculate development period per country ──

def calc_dev_period(country_df):
    """Estimate development period (years) based on project pipeline."""
    has_prod = (country_df['Status'] == 'Production').any()
    has_dev = (country_df['Status'].isin(['Development', 'Feasibility'])).any()
    if has_prod:
        return 8   # established producer, shorter permitting
    elif has_dev:
        return 12  # pipeline exists
    else:
        return 15  # greenfield country


# ── 3.3 Build the factors DataFrame ──

factors_rows = []

for i, country in enumerate(countries):
    c_df = df[df['Country (Short Form)'] == country]
    params = country_params[country]
    
    t = params['tonnage']
    g = params['grade']
    
    row = {
        'INDEX': i,
        'WEIGHTING': 0.1,
        'GEOPACKAGE_REGION_COLUMN': 'REGION',
        'REGION': country,
        'DEPOSIT_TYPE': 'Flake graphite',
        'COMMODITY_PRIMARY': 'graphite',
        
        # Grade distribution (lognormal, mass fraction)
        'GRADE_MODEL': 'lognormal',
        'GRADE_A': round(g['mu'], 6),
        'GRADE_B': round(g['sigma'], 6),
        'GRADE_C': round(g['max'], 6),
        'GRADE_D': 0,
        
        # Tonnage distribution (lognormal, ore tonnes)
        'TONNAGE_MODEL': 'lognormal',
        'TONNAGE_A': round(t['mu'], 6),
        'TONNAGE_B': round(t['sigma'], 6),
        'TONNAGE_C': round(t['max'], 0),
        'TONNAGE_D': 0,
        
        # Brownfield
        'BROWNFIELD_TONNAGE_FACTOR': 0.05,
        'BROWNFIELD_GRADE_FACTOR': 1,
        
        # Capacity (from global fit — concentrate t/yr)
        'CAPACITY_BASIS': 0,
        'CAPACITY_A': round(cap_fit['median'], 2),
        'CAPACITY_B': 0,
        'CAPACITY_SIGMA': round(cap_fit['sigma'] * 0.5, 4),  # narrower for new mines
        'CAPACITY_SIGMA_LOG10': np.nan,
        
        # Mine life
        'LIFE_MIN': 10,
        'LIFE_MAX': 40,
        
        # Recovery
        'RECOVERY': 0.85,
        
        # Revenue — placeholder
        'REVENUE_MODEL': 'contained_value',
        'REVENUE_A': 0,
        'REVENUE_B': 0,
        'REVENUE_C': 0,
        'REVENUE_D': 0,
        
        # Cost — placeholder
        'COST_MODEL': 'contained_value',
        'COST_A': 0,
        'COST_B': 0,
        'COST_C': 0,
        'COST_D': 0,
        
        # Mine cost — placeholder
        'MINE_COST_MODEL': 'fixed',
        'MINE_COST_A': 0,
        'MINE_COST_B': 0,
        'MINE_COST_C': 0,
        'MINE_COST_D': 0,
        
        # Development
        'DEVELOPMENT_PERIOD': calc_dev_period(c_df),
        'DEVELOPMENT_PROBABILITY': calc_dev_probability(c_df),
        
        # Coproduct — none for graphite
        'COPRODUCT_COMMODITY': np.nan,
        'COPRODUCT_GRADE_MODEL': 'contained_value',
        'COPRODUCT_A': 0,
        'COPRODUCT_B': 0,
        'COPRODUCT_C': 0,
        'COPRODUCT_D': 0,
        'COPRODUCT_RECOVERY': 0,
        'COPRODUCT_SUPPLY_TRIGGER': 0,
        'COPRODUCT_BROWNFIELD_GRADE_FACTOR': 0,
        'COPRODUCT_REVENUE_MODEL': 'contained_value',
        'COPRODUCT_REVENUE_A': 0,
        'COPRODUCT_REVENUE_B': 0,
        'COPRODUCT_REVENUE_C': 0,
        'COPRODUCT_REVENUE_D': 0,
        'COPRODUCT_COST_MODEL': 'contained_value',
        'COPRODUCT_COST_A': 0,
        'COPRODUCT_COST_B': 0,
        'COPRODUCT_COST_C': 0,
        'COPRODUCT_COST_D': 0,
        
        # Label
        'LABEL': f'{country} - Flake graphite',
    }
    factors_rows.append(row)

epf = pd.DataFrame(factors_rows)
print(f"Built exploration_production_factors: {len(epf)} rows (1 per country)")

Built exploration_production_factors: 32 rows (1 per country)


In [9]:
# ── 3.4 Summary of fitted distributions ──

print(f"{'='*90}")
print(f"DISTRIBUTION SUMMARY BY COUNTRY")
print(f"{'='*90}")
print(f"{'Country':<20} {'Ton src':<8} {'Ton median':>12} {'Grd src':<8} "
      f"{'Grd median':>10} {'Dev prob':>9} {'Dev period':>10}")
print("-" * 90)

for _, row in epf.iterrows():
    country = row['REGION']
    params = country_params[country]
    t_med = np.exp(row['TONNAGE_A'])
    g_med = np.exp(row['GRADE_A']) * 100  # back to %

    print(f"{country:<20} {params['t_source']:<8} {t_med/1e6:>10.1f} Mt "
          f"{params['g_source']:<8} {g_med:>9.1f}% "
          f"{row['DEVELOPMENT_PROBABILITY']:>9.3f} {int(row['DEVELOPMENT_PERIOD']):>8} yr")

DISTRIBUTION SUMMARY BY COUNTRY
Country              Ton src    Ton median Grd src  Grd median  Dev prob Dev period
------------------------------------------------------------------------------------------
Afghanistan          global         15.1 Mt global         7.8%     0.050       15 yr
Australia            country        17.6 Mt country        8.3%     0.050       15 yr
Bhutan               global         15.1 Mt global         7.8%     0.050       12 yr
Brazil               global         15.1 Mt global         7.8%     0.667        8 yr
Canada               country        19.5 Mt country        5.6%     0.083        8 yr
Chad                 global         15.1 Mt global         7.8%     0.050       15 yr
China                country        75.8 Mt country        8.4%     0.407        8 yr
Ethiopia             global         15.1 Mt global         7.8%     0.050       15 yr
Greenland            global         15.1 Mt global         7.8%     0.050       15 yr
Guinea             

In [10]:
# ── 3.5 Save input_exploration_production_factors.csv ──

epf.to_csv('./outputs/input_exploration_production_factors.csv', index=False)
print(f"Saved input_exploration_production_factors.csv: {len(epf)} rows")

Saved input_exploration_production_factors.csv: 32 rows


In [11]:
# ══════════════════════════════════════════════════════════════════════════════
# 4. VALIDATION — Sanity check the fitted distributions
#
# Simulate draws from each country's lognormal distributions and compare
# against actual observed data.
# ══════════════════════════════════════════════════════════════════════════════
import random
random.seed(42)

print(f"{'='*70}")
print(f"VALIDATION: Simulated vs Observed")
print(f"{'='*70}")

# Countries with enough data to validate
validate_countries = [c for c in countries
                      if country_params[c]['t_source'] == 'country']

for country in validate_countries:
    c_res = res[res['Country (Short Form)'] == country]
    params = country_params[country]
    t = params['tonnage']
    g = params['grade']
    
    # Simulate 1000 draws
    sim_tonnage = [min(abs(random.lognormvariate(t['mu'], t['sigma'])), t['max'])
                   for _ in range(1000)]
    sim_grade = [min(abs(random.lognormvariate(g['mu'], g['sigma'])), g['max'])
                 for _ in range(1000)]
    
    obs_ton = c_res['Total Resource (mt)'].dropna()
    obs_grd = c_res['grade_frac'].dropna()
    
    print(f"\n  {country} (n={len(c_res)} deposits):")
    print(f"    Tonnage (Mt):  observed median={obs_ton.median()/1e6:.1f}, "
          f"simulated median={np.median(sim_tonnage)/1e6:.1f}, "
          f"observed range=[{obs_ton.min()/1e6:.1f}, {obs_ton.max()/1e6:.1f}], "
          f"simulated 5-95%=[{np.percentile(sim_tonnage, 5)/1e6:.1f}, "
          f"{np.percentile(sim_tonnage, 95)/1e6:.1f}]")
    if not obs_grd.empty:
        print(f"    Grade (TGC%):  observed median={obs_grd.median()*100:.1f}%, "
              f"simulated median={np.median(sim_grade)*100:.1f}%, "
              f"observed range=[{obs_grd.min()*100:.1f}%, {obs_grd.max()*100:.1f}%], "
              f"simulated 5-95%=[{np.percentile(sim_grade, 5)*100:.1f}%, "
              f"{np.percentile(sim_grade, 95)*100:.1f}%]")

VALIDATION: Simulated vs Observed

  Australia (n=18 deposits):
    Tonnage (Mt):  observed median=13.2, simulated median=17.7, observed range=[3.1, 434.5], simulated 5-95%=[1.8, 169.1]
    Grade (TGC%):  observed median=7.4%, simulated median=8.4%, observed range=[4.4%, 16.1%], simulated 5-95%=[4.2%, 16.1%]

  Canada (n=12 deposits):
    Tonnage (Mt):  observed median=23.9, simulated median=19.2, observed range=[2.3, 142.8], simulated 5-95%=[1.6, 142.8]
    Grade (TGC%):  observed median=5.4%, simulated median=5.6%, observed range=[1.8%, 17.2%], simulated 5-95%=[1.4%, 17.2%]

  China (n=3 deposits):
    Tonnage (Mt):  observed median=28.3, simulated median=68.7, observed range=[22.6, 680.0], simulated 5-95%=[3.2, 680.0]
    Grade (TGC%):  observed median=9.9%, simulated median=8.8%, observed range=[3.8%, 16.0%], simulated 5-95%=[2.5%, 16.0%]

  Malawi (n=3 deposits):
    Tonnage (Mt):  observed median=8.6, simulated median=7.1, observed range=[2.7, 14.5], simulated 5-95%=[1.7, 14.5]
 

In [12]:
# ══════════════════════════════════════════════════════════════════════════════
# 5. FINAL SUMMARY
# ══════════════════════════════════════════════════════════════════════════════

print(f"{'='*70}")
print(f"PEMMSS INPUT FILES GENERATED")
print(f"{'='*70}")

print(f"\n1. input_projects.csv")
print(f"   Records:   {len(projects)}")
print(f"   Countries: {projects['REGION'].nunique()}")
print(f"   Status 2 (Producing):   {(projects['STATUS']==2).sum()}")
print(f"   Status 1 (Developed):   {(projects['STATUS']==1).sum()}")
print(f"   Status 0 (Undeveloped): {(projects['STATUS']==0).sum()}")
print(f"   With resource data:     {projects['REMAINING_RESOURCE'].notna().sum()}")
print(f"   With capacity data:     {projects['PRODUCTION_CAPACITY'].notna().sum()}")
print(f"   With grade data:        {projects['GRADE'].notna().sum()}")

print(f"\n2. input_exploration_production_factors.csv")
print(f"   Regions (countries):    {len(epf)}")
print(f"   Deposit type:           Flake graphite (all)")
print(f"   Country-fitted tonnage: {sum(1 for c in countries if country_params[c]['t_source']=='country')}")
print(f"   Country-fitted grade:   {sum(1 for c in countries if country_params[c]['g_source']=='country')}")
print(f"   Global fallback:        {sum(1 for c in countries if country_params[c]['t_source']=='global')}")

print(f"\n   PLACEHOLDERS to fill before running PEMMSS:")
print(f"   - REVENUE_MODEL / REVENUE_A-D (currently 'contained_value' / 0)")
print(f"   - COST_MODEL / COST_A-D (currently 'contained_value' / 0)")
print(f"   - MINE_COST_MODEL / MINE_COST_A-D (currently 'fixed' / 0)")
print(f"   - WEIGHTING values (currently 0.1 for all)")

PEMMSS INPUT FILES GENERATED

1. input_projects.csv
   Records:   237
   Countries: 32
   Status 2 (Producing):   31
   Status 1 (Developed):   32
   Status 0 (Undeveloped): 174
   With resource data:     96
   With capacity data:     41
   With grade data:        97

2. input_exploration_production_factors.csv
   Regions (countries):    32
   Deposit type:           Flake graphite (all)
   Country-fitted tonnage: 9
   Country-fitted grade:   9
   Global fallback:        23

   PLACEHOLDERS to fill before running PEMMSS:
   - REVENUE_MODEL / REVENUE_A-D (currently 'contained_value' / 0)
   - COST_MODEL / COST_A-D (currently 'contained_value' / 0)
   - MINE_COST_MODEL / MINE_COST_A-D (currently 'fixed' / 0)
   - WEIGHTING values (currently 0.1 for all)
